In [48]:
import shutil
from pathlib import Path

from tqdm.auto import tqdm

### Identify vsi/ets pairs for conversion

In [62]:
# root = Path("/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology")
root = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/")

# mouse dirs ≥ 7
mouse_dirs = sorted(
    [p for p in root.glob("mouse_*") if int(p.name.removeprefix("mouse_")) >= 7],
    key=lambda p: int(p.name.removeprefix("mouse_")),
)

# largest .ets per mouse
largest_ets_per_mouse = []
for mouse_dir in mouse_dirs:
    ets_files = list(mouse_dir.rglob("*.ets"))
    if ets_files:
        largest_ets_per_mouse.append(max(ets_files, key=lambda f: f.stat().st_size))

# map each .ets to its sibling .vsi using the 2nd parent folder name
def ets_to_vsi(ets_path: Path) -> Path:
    acq_dir = ets_path.parent.parent.name              # e.g. "_20250901_..._5555_"
    acq_base = acq_dir.strip("_")                      # -> "20250901_..._5555"
    mouse_dir = ets_path.parents[2]     # .../<mouse_N>/
    return mouse_dir / f"{acq_base}.vsi"               # .../<mouse_N>/<acq_base>.vsi

corresponding_vsi = []
for ets in largest_ets_per_mouse:
    vsi = ets_to_vsi(ets)
    corresponding_vsi.append(vsi if vsi.exists() else None)

# show pairs (and which are missing)
for ets, vsi in zip(largest_ets_per_mouse, corresponding_vsi):
    print(f"{ets}  ->  {vsi if vsi else 'NO MATCH'}")


/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/_20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557_/stack1/frame_t_0.ets  ->  /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.vsi
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/_20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549_/stack1/frame_t_0.ets  ->  /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.vsi
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/_20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550_/stack1/frame_t_0.ets  ->  NO MATCH
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_loc

### Move ets and vsi to local folder so that I can write out to tiff with permission

In [53]:
# move or copy function — set move=True to move instead of copy
def move_file_preserve_structure(src: Path):
    rel_path = src.relative_to(root)             # path below Histology/
    dest_path = dest_root / rel_path             # preserve hierarchy
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(src, dest_path)
    return dest_path

In [57]:
largest_ets_per_mouse[1:]

[PosixPath('/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/mouse_8/_20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549_/stack1/frame_t_0.ets'),
 PosixPath('/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/mouse_9/_20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550_/stack1/frame_t_0.ets'),
 PosixPath('/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology/mouse_11/_20250901_40X_TimerMtb_BP_mice11_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250902_5555_/stack1/frame_t_0.ets')]

In [ ]:
dest_root = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice")

# iterate over pairs
for ets, vsi in tqdm(zip(largest_ets_per_mouse[2:], corresponding_vsi[2:]), total = len(corresponding_vsi)):
    if not ets or not vsi:
        continue  # skip incomplete pairs
    
    try:
        new_ets = move_file_preserve_structure(ets)
    except:
        print(ets, 'failed?')
    try:
        new_vsi = move_file_preserve_structure(vsi)
    except:
        print(vsi, 'failed?')
    # print(f"✅ {ets.name} and {vsi.name} -> {new_ets.parent}")

## Convert to tiff

In [64]:
output_fns = []
for input_fn in tqdm(corresponding_vsi):
    try:
        output_fn = input_fn.with_suffix('.ome.tif')
        output_fns.append(output_fn)
        !/home/dayn/miniconda3/envs/godspee/bin/bfconvert \
        -bigtiff -compression LZW \
        -tilex 512 -tiley 512 \
        -pyramid-resolutions 5 \
        -pyramid-scale 2 \
        "$input_fn" "$output_fn"
    except:
        print(input_fn, 'failed?')

  0%|          | 0/4 [00:00<?, ?it/s]

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.vsi
CellSensReader initializing /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.vsi
[CellSens VSI] -> /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.ome.tif [OME-TIFF]
Tile size = 512 x 512
	Series 0: converted 1/33 planes (3%)
	Series 0: converted 2/33 planes (6%)
	Series 0: converted 3/33 planes (9%)
	Series 0: converted 4/33 planes (12%)
	Series 0: converted 5/33 planes (15%)
	Series 0: converted 6/33 planes (18%)
	Series 0: converted 7/33 planes (21%)
	Series 0: converted 8/33 planes (24%)
	Series 0: converted 9/33 planes (27%)
	Series 0: converted 10/33 planes (

## Do 2 more positions conversion to tiled pyramidal tiffs prior to full zarr conversion

In [65]:
# root = Path("/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology")
root = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/")

# mouse dirs ≥ 7
mouse_dirs = sorted(
    [p for p in root.glob("mouse_*") if int(p.name.removeprefix("mouse_")) >= 7],
    key=lambda p: int(p.name.removeprefix("mouse_")),
)

# largest .ets per mouse
largest_ets_per_mouse = []
for mouse_dir in mouse_dirs:
    ets_files = list(mouse_dir.rglob("*.ets"))
    if ets_files:
        largest_ets_per_mouse.append(max(ets_files, key=lambda f: f.stat().st_size))

# map each .ets to its sibling .vsi using the 2nd parent folder name
def ets_to_vsi(ets_path: Path) -> Path:
    acq_dir = ets_path.parent.parent.name              # e.g. "_20250901_..._5555_"
    acq_base = acq_dir.strip("_")                      # -> "20250901_..._5555"
    mouse_dir = ets_path.parents[2]     # .../<mouse_N>/
    return mouse_dir / f"{acq_base}.vsi"               # .../<mouse_N>/<acq_base>.vsi

corresponding_vsi = []
for ets in largest_ets_per_mouse:
    vsi = ets_to_vsi(ets)
    corresponding_vsi.append(vsi if vsi.exists() else None)

# show pairs (and which are missing)
for ets, vsi in zip(largest_ets_per_mouse, corresponding_vsi):
    print(f"{ets}  ->  {vsi if vsi else 'NO MATCH'}")


/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/_20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557_/stack1/frame_t_0.ets  ->  /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_7/20250901_40X_TimerMtb_BP_mice5_mice6_mice7_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250903_5557.vsi
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/_20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549_/stack1/frame_t_0.ets  ->  /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_8/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5549.vsi
/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/_20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550_/stack1/frame_t_0.ets  ->  /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation

In [ ]:
output_fns = []
for input_fn in tqdm(corresponding_vsi[2:]):
    try:
        output_fn = input_fn.with_suffix('.ome.tif')
        output_fns.append(output_fn)
        !/home/dayn/miniconda3/envs/godspee/bin/bfconvert \
        -bigtiff -compression LZW \
        -tilex 512 -tiley 512 \
        -pyramid-resolutions 5 \
        -pyramid-scale 2 \
        "$input_fn" "$output_fn"
    except:
        print(input_fn, 'failed?')

  0%|          | 0/2 [00:00<?, ?it/s]

/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550.vsi
CellSensReader initializing /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550.vsi
[CellSens VSI] -> /mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/mouse_9/20250821_40X_TimerMtb_BP_mice8_mice9_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250822_5550.ome.tif [OME-TIFF]
Tile size = 512 x 512
	Series 0: converted 1/33 planes (3%)
	Series 0: converted 2/33 planes (6%)
	Series 0: converted 3/33 planes (9%)
	Series 0: converted 4/33 planes (12%)
	Series 0: converted 5/33 planes (15%)
	Series 0: converted 6/33 planes (18%)
	Series 0: converted 7/33 planes (21%)
	Series 0: converted 8/33 planes (24%)
	Series 0: converted 9/33 planes (27%)
	Series 0: converted 10/33 planes (30%)
	Series 0: co

In [ ]:
# root = Path("/mnt/NEMO/home/shared/Shared - Baptiste Pradel/Histology")
root = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/")

# mouse dirs ≥ 7
mouse_dirs = sorted(
    [p for p in root.glob("mouse_*") if int(p.name.removeprefix("mouse_")) >= 7],
    key=lambda p: int(p.name.removeprefix("mouse_")),
)

# largest .ets per mouse
largest_ets_per_mouse = []
for mouse_dir in mouse_dirs:
    ets_files = list(mouse_dir.rglob("*.ets"))
    if ets_files:
        largest_ets_per_mouse.append(max(ets_files, key=lambda f: f.stat().st_size))

# map each .ets to its sibling .vsi using the 2nd parent folder name
def ets_to_vsi(ets_path: Path) -> Path:
    acq_dir = ets_path.parent.parent.name              # e.g. "_20250901_..._5555_"
    acq_base = acq_dir.strip("_")                      # -> "20250901_..._5555"
    mouse_dir = ets_path.parents[2]     # .../<mouse_N>/
    return mouse_dir / f"{acq_base}.vsi"               # .../<mouse_N>/<acq_base>.vsi

corresponding_vsi = []
for ets in largest_ets_per_mouse:
    vsi = ets_to_vsi(ets)
    corresponding_vsi.append(vsi if vsi.exists() else None)

# show pairs (and which are missing)
for ets, vsi in zip(largest_ets_per_mouse, corresponding_vsi):
    print(f"{ets}  ->  {vsi if vsi else 'NO MATCH'}")


In [ ]:
output_fns = []
for input_fn in tqdm(corresponding_vsi):

    output_fn = input_fn.with_suffix('.ome.tif')
    output_fns.append(output_fn)

## Convert to Zarr

In [ ]:
import dask.array as da
import tifffile
import zarr
from dask.diagnostics import ProgressBar
from ome_zarr.io import parse_url
from ome_zarr.writer import (
    add_metadata,
    write_image,
    write_multiscales_metadata,
)

In [ ]:
for output_fn in tqdm(output_fns):
    zarr_image = tifffile.imread(output_fn, aszarr=True)
    dask_image = da.from_zarr(zarr_image)
    print(output_fn, dask_image.shape)
    dask_image = dask_image.transpose(1, 0, 2, 3,)  # lazy; no data copy
    # v0.5 / Zarr v3 store
    out_zarr = output_fn.with_suffix('.zarr')

    store = parse_url(out_zarr, mode="w").store
    root = zarr.group(store=store)
    
    # writes data + builds a 2x YX pyramid by default
    with ProgressBar():  # optional live progress
        write_image(
            image=dask_image,
            group=root,
            axes="czyx",
            storage_options=dict(chunks=(1, 1, 512, 512)),  # (C,Z,Y,X)
        )
    
    # optional: channel labels for nicer viewing in napari/viv
    add_metadata(root, {"omero": {
        "channels": [
            {"label": "CF405"},
            {"label": "CF488"},
            {"label": "CF561"},
        ]
    }})

    # after write_image(... axes="czyx")
    level_names = sorted(root.array_keys(), key=int)   # <-- not group_keys()
    
    axes = [
        {"name": "c", "type": "channel"},
        {"name": "z", "type": "space", "unit": "micrometer"},
        {"name": "y", "type": "space", "unit": "micrometer"},
        {"name": "x", "type": "space", "unit": "micrometer"},
    ]
    
    px_z, px_y, px_x = 2.0, 0.1625, 0.1625
    datasets = []
    for i, p in enumerate(level_names):
        datasets.append({
            "path": p,
            "coordinateTransformations": [
                {"type": "scale", "scale": [1.0, px_z, px_y*(2**i), px_x*(2**i)]},  # C Z Y X
                {"type": "translation", "translation": [0, 0, 0, 0]},
            ]
        })
    
    write_multiscales_metadata(root, datasets=datasets, axes=axes)


# Arx

In [ ]:
import time
from datetime import datetime, timedelta
from pathlib import Path

# ---- config ----
ref_path = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/_20250814_40X_TimerMtb_BP_mice10_DAPI_TimerG_TimerR_Multichannel Z-Stack_20250814_5375_/stack1/frame_t_0.ets")
out_path = Path("/mnt/OPERA3/Nathan/data/macrohet/mtb_tissue_localisation/whole_slice/exports/mtb_slice_5375_s0.ome.tif")

check_every_s = 90           # polling interval
min_samples_for_eta = 5      # wait this many samples before showing ETA
window_samples = 5           # moving-average window for rate
stall_threshold_gb = 0.01    # consider “no progress” if growth < this per interval
stall_checks = 60             # consecutive stalls before we say “probably done”
target_equals_ref = True     # if False, don’t compare to ref size; just use stall logic
# -----------------

def gb(bytes_): return bytes_ / (1024**3)

ref_size_gb = gb(ref_path.stat().st_size)
print(f"Reference size: {ref_size_gb:.2f} GB (for context)\n")

sizes = []
times = []
no_progress = 0

while True:
    if out_path.exists():
        s = gb(out_path.stat().st_size)
        t = time.time()
        sizes.append(s); times.append(t)

        # progress %
        pct_str = f"{(s/ref_size_gb*100):.1f}%" if ref_size_gb > 0 else "—"

        # compute smoothed rate after we have enough points
        eta_str = "estimating…"
        rate_gb_per_min = 0.0
        if len(sizes) >= min_samples_for_eta:
            w = sizes[-window_samples:]
            wt = times[-window_samples:]
            ds = w[-1] - w[0]
            dt_min = (wt[-1] - wt[0]) / 60
            if dt_min > 0 and ds > 0:
                rate_gb_per_min = ds / dt_min

                if target_equals_ref:
                    rem = max(ref_size_gb - s, 0.0)
                    if rate_gb_per_min > 0:
                        minutes = rem / rate_gb_per_min
                        eta_time = datetime.now() + timedelta(minutes=minutes)
                        eta_str = eta_time.strftime("%H:%M:%S")
                else:
                    eta_str = "—"  # unknown final size; stall detection will stop loop

        # detect stalls (don’t trigger on first sample)
        if len(sizes) >= 2:
            delta = sizes[-1] - sizes[-2]
            if delta < stall_threshold_gb:
                no_progress += 1
            else:
                no_progress = 0

        # status line
        print(
            f"\r{datetime.now().strftime('%H:%M:%S')} | "
            f"{s:7.2f} GB ({pct_str}) | "
            f"{rate_gb_per_min:5.2f} GB/min | ETA {eta_str} | "
            f"stalls {no_progress}/{stall_checks}",
            end=""
        )

        # stop conditions:
        done_by_size = target_equals_ref and (s >= 0.995 * ref_size_gb)
        done_by_stall = no_progress >= stall_checks
        if done_by_size or done_by_stall:
            print("\n✅ Done (size target reached or growth stalled).")
            break
    else:
        print("\rWaiting for output file...", end="")

    time.sleep(check_every_s)
    

Reference size: 167.93 GB (for context)

10:39:02 |   82.98 GB (49.4%) |  0.00 GB/min | ETA estimating… | stalls 0/60